In [ ]:
import pandas as pd
import numpy as np
import os

np.random.seed(42)



# Customer Table

In [2]:
n = 500000

# Step 1: Create equal segments
segments = np.random.choice(['Low', 'Medium', 'High'], size=n, p=[0.50, 0.35, 0.15])
    
cust_df = pd.DataFrame({
    'customer_id': np.arange(1, n + 1),
    'segment': segments
})

# Step 2: Assign offers WITHIN each segment
offers = []

for seg in ['High', 'Medium', 'Low']:
    seg_size = (cust_df['segment'] == seg).sum()
    
    # 10% control (offer 4), 30% each for 1,2,3
    seg_offers = (
        ['Control'] * int(0.10 * seg_size) +
        ['Discount'] * int(0.30 * seg_size) +
        ['Free Trial'] * int(0.30 * seg_size) +
        ['Early Access'] * int(0.30 * seg_size)
    )
    
    # Adjust for rounding
    while len(seg_offers) < seg_size:
        seg_offers.append(np.random.choice(['Discount', 'Free Trial', 'Early Access']))
    
    np.random.shuffle(seg_offers)
    offers.extend(seg_offers)

cust_df['offer'] = offers

# Step 3: Generate avg_weekly_hours_pre based on segment
choices = [
    np.random.lognormal(0.5, 0.5, n),
    np.random.lognormal(1.5, 0.5, n),
    np.random.lognormal(2.5, 0.5, n),
]
cust_df['avg_weekly_hours_pre'] = np.select(
    condlist=[
        cust_df['segment'] == 'Low',
        cust_df['segment'] == 'Medium',
        cust_df['segment'] == 'High'
    ],
    choicelist=choices,
    default=np.nan
)

cust_df.head()

,customer_id,segment,offer,avg_weekly_hours_pre
0,1,Low,Free Trial,1.205087
1,2,High,Free Trial,9.673556
2,3,Medium,Free Trial,3.064467
3,4,Medium,Discount,9.936197
4,5,Low,Discount,0.744099


In [3]:
cust_df['offer'].value_counts()

offer
Free Trial      150001
Early Access    150001
Discount        150000
Control          49998
Name: count, dtype: int64

In [4]:
cust_df['segment'].value_counts()

segment
Low       249555
Medium    175398
High       75047
Name: count, dtype: int64

In [5]:
cust_df[['segment', 'offer']].value_counts()

segment  offer       
Low      Free Trial      74967
         Discount        74962
         Early Access    74858
Medium   Early Access    52619
         Discount        52570
         Free Trial      52524
Low      Control         24768
High     Early Access    22524
         Free Trial      22510
         Discount        22468
Medium   Control         17685
High     Control          7545
Name: count, dtype: int64

In [6]:
print(cust_df['segment'].value_counts(normalize=True).round(3))
print(cust_df.groupby('segment')['offer'].value_counts(normalize=True).round(3))

segment
Low       0.499
Medium    0.351
High      0.150
Name: proportion, dtype: float64
segment  offer       
High     Early Access    0.300
         Free Trial      0.300
         Discount        0.299
         Control         0.101
Low      Free Trial      0.300
         Discount        0.300
         Early Access    0.300
         Control         0.099
Medium   Early Access    0.300
         Discount        0.300
         Free Trial      0.299
         Control         0.101
Name: proportion, dtype: float64


In [7]:
print(cust_df.groupby('segment')['avg_weekly_hours_pre'].describe().round(2))

            count   mean   std   min   25%    50%    75%     max
segment                                                         
High      75047.0  13.81  7.34  1.53  8.71  12.20  17.08  104.73
Low      249555.0   1.87  0.99  0.16  1.18   1.65   2.31   16.27
Medium   175398.0   5.07  2.70  0.42  3.19   4.48   6.27   39.90


In [ ]:
os.makedirs('../data/raw', exist_ok=True)
cust_df.to_csv('../data/raw/customers.csv', index=False)